In [8]:
import pandas as pd

# Define the paths to your original data files
ig_data_path = 'C:/Users/Ben/Documents/crowdsourcing/original/IG_Data.csv'
xs_data_path = 'C:/Users/Ben/Documents/crowdsourcing/original/XSIndiegogoProduction_updated.csv'


# Load the two CSV files into pandas DataFrames
try:
    df_ig = pd.read_csv(ig_data_path)
    # Use low_memory=False to handle potential mixed data types
    df_xs = pd.read_csv(xs_data_path, low_memory=False)

    print(f"Loaded {len(df_ig):,} rows from IG_Data.csv")
    print(f"Loaded {len(df_xs):,} rows from XSIndiegogoProduction_updated.csv")

    # --- Standardize ID columns to the same data type ---
    print("\n--- Standardizing ID data types ---")
    
    # Convert 'project_id' to nullable integer
    df_ig['project_id'] = pd.to_numeric(df_ig['project_id'], errors='coerce').astype('Int64')
    print(f"Data type of 'project_id' is now: {df_ig['project_id'].dtype}")

    # Convert 'id' to nullable integer
    df_xs['id'] = pd.to_numeric(df_xs['id'], errors='coerce').astype('Int64')
    print(f"Data type of 'id' is now: {df_xs['id'].dtype}")

    # Drop rows where the ID is now missing after conversion
    df_ig.dropna(subset=['project_id'], inplace=True)
    df_xs.dropna(subset=['id'], inplace=True)

    # --- Perform the Inner Join with Standardized Keys ---
    merged_df = pd.merge(
        df_ig,
        df_xs,
        left_on='project_id',
        right_on='id',
        how='inner'
    )

    print(f"\nSuccessfully performed an inner join with standardized keys.")
    print(f"The merged DataFrame contains {len(merged_df):,} rows.")
    if len(merged_df) == 0:
        print("--> CONFIRMED: There is zero overlap between the two files.")
    else:
        print(f"--> FOUND: There are {len(merged_df):,} projects that exist in both source files.")


except FileNotFoundError as e:
    print(f"Error: Could not find a file. Please check the path: {e.filename}")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded 17,257 rows from IG_Data.csv
Loaded 62,671 rows from XSIndiegogoProduction_updated.csv

--- Standardizing ID data types ---
Data type of 'project_id' is now: Int64
Data type of 'id' is now: Int64

Successfully performed an inner join with standardized keys.
The merged DataFrame contains 0 rows.
--> CONFIRMED: There is zero overlap between the two files.


In [9]:
# --- Compare Key Columns ---

# Check if the merged_df was created successfully before proceeding
if 'merged_df' in locals() and not merged_df.empty:
    print("--- Comparing 'funds_raised_percent' (from IG_Data) vs. 'goal_percentage' (from XS_Data) ---\n")

    # Define the columns to compare
    compare_cols = ['funds_raised_percent', 'goal_percentage', 'state']

    # Show a side-by-side comparison for the first 10 rows
    print("Side-by-side comparison (first 10 rows):")
    print(merged_df[compare_cols].head(10))

    # Show summary statistics for both columns to check for systematic differences
    print("\n\nSummary statistics:")
    # We need to handle the '%' sign in 'goal_percentage' before converting to numeric
    merged_df['goal_percentage_numeric'] = merged_df['goal_percentage'].str.replace('%', '', regex=False).astype(float)
    
    stats = merged_df[['funds_raised_percent', 'goal_percentage_numeric']].describe()
    print(stats)
    
    print("\n\n--- Comparing 'state' column with a calculated success indicator ---")
    
    # Create a success indicator from 'goal_percentage'
    merged_df['success_from_goal_perc'] = (merged_df['goal_percentage_numeric'] >= 100).astype(int)
    
    # Create a crosstab to see how 'state' aligns with our new indicator
    alignment_table = pd.crosstab(merged_df['state'], merged_df['success_from_goal_perc'])
    print("\nAlignment of 'state' with success calculated from 'goal_percentage':")
    print(alignment_table)

else:
    print("Skipping column comparison because the merged DataFrame is empty or does not exist.")

# New Diagnostic Cell: Inspect the Join Keys

print("--- Investigating the Join Keys ('project_id' and 'id') ---\n")

# Check if the dataframes were loaded
if 'df_ig' in locals() and 'df_xs' in locals():
    
    print("--- Data Types ---")
    print(f"Type of 'project_id' in IG_Data: {df_ig['project_id'].dtype}")
    print(f"Type of 'id' in XS_Data: {df_xs['id'].dtype}")
    
    print("\n--- Sample IDs from IG_Data.csv ---")
    print(df_ig['project_id'].head(10).to_list())
    
    print("\n--- Sample IDs from XSIndiegogoProduction_updated.csv ---")
    print(df_xs['id'].head(10).to_list())
    
    # Advanced check: Convert to sets and find the intersection
    try:
        set_ig_ids = set(df_ig['project_id'])
        set_xs_ids = set(df_xs['id'])
        intersection_count = len(set_ig_ids.intersection(set_xs_ids))
        
        print(f"\n--- Intersection Check ---")
        print(f"Found {intersection_count} common IDs between the two files.")
    except Exception as e:
        print(f"\nCould not perform set intersection check. Error: {e}")

else:
    print("One or both DataFrames were not loaded. Please run the first cell again.")

Skipping column comparison because the merged DataFrame is empty or does not exist.
--- Investigating the Join Keys ('project_id' and 'id') ---

--- Data Types ---
Type of 'project_id' in IG_Data: Int64
Type of 'id' in XS_Data: Int64

--- Sample IDs from IG_Data.csv ---
[2684556, 2726165, 2821770, 2822666, 2646452, 2696660, 2643903, 2801544, 2734171, 2799629]

--- Sample IDs from XSIndiegogoProduction_updated.csv ---
[250739, 252403, 281198, 251776, 252287, 271821, 281017, 259042, 265086, 276617]

--- Intersection Check ---
Found 0 common IDs between the two files.


In [ ]:
# --- Test New Success Indicator on Full XS_Data File ---

# This cell tests the core hypothesis: Can 'goal_percentage' from the XS_Data file
# solve our success indicator problem for Indiegogo projects?

print("="*80)
print("Testing the impact of using 'goal_percentage' on the full XSIndiegogoProduction_updated.csv dataset.")

# Check if the df_xs was loaded successfully
if 'df_xs' in locals() and not df_xs.empty:
    
    # --- Step 1: Create the new success indicator ---
    # Convert 'goal_percentage' to a numeric format, coercing errors to NaN
    df_xs['goal_percentage_numeric'] = pd.to_numeric(df_xs['goal_percentage'].str.replace('%', '', regex=False), errors='coerce')

    # Create a new success indicator based on the numeric percentage
    # We'll also consider the 'state' column as a potential indicator of success
    df_xs['new_success_indicator'] = (
        (df_xs['goal_percentage_numeric'] >= 100) |
        (df_xs['state'] == 'successful')
    ).astype(int)
    
    
    # --- Step 2: Evaluate the impact ---
    num_total_projects = len(df_xs)
    num_successful_projects = df_xs['new_success_indicator'].sum()
    
    if num_total_projects > 0:
        success_rate = (num_successful_projects / num_total_projects) * 100
        print(f"\nTotal projects in XSIndiegogoProduction_updated.csv: {num_total_projects:,}")
        print(f"Number of successful projects using the new indicator: {num_successful_projects:,}")
        print(f"New success rate for Indiegogo projects: {success_rate:.2f}%")
        
        print("\n--- Breakdown of Success by 'state' Column ---")
        print(df_xs['state'].value_counts(dropna=False))

    else:
        print("The XSIndiegogoProduction_updated.csv dataframe is empty.")

else:
    print("Skipping the final test because the df_xs DataFrame does not exist.")

print("="*80)

